In [290]:
import pandas as pd

In [291]:
student_observations = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='Student_Observations')
class_plan = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='Class_Plan',index_col='class_num')
kc_nodes = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='KC_Nodes')
hwk_items = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='Homework_Items',index_col='class_num')

Need the following columns :
| DataShop col | Data col | File |
|--------------|----------|------|
|Anon Student ID | student_id | student_observations |
|Problem Hierarchy | reporting_group | kc_nodes |
|Problem name | source_question | student_observations |
|Step name | observation_id | student_observations |
|First Attempt |score | student_observations |
|Incorrects|score | student_observations |
|Corrects|score | student_observations |
|Hints|(maybe use the partial corrects?) | - | 
|KC (Default) | kc_label | kc_nodes |
|Opportunity (Default)| class_num | student_observations |

## Preprocessing

### Student Observations

In [292]:
student_observations.head()

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level
0,S001,HW1,1,HW1_PCA_Q01,MCQ,PCA Q01,KC.U1.02.observational_unit_variable,KC.U1.02.observational_unit_variable,0.0,1,0,C,E,NaN
1,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.0,1,100,E,E,NaN
2,S001,HW1,1,HW1_PCA_Q03,MCQ,PCA Q03,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,0.0,1,0,C,D,NaN
3,S001,HW1,1,HW1_PCA_Q04,MCQ,PCA Q04,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,E,E,NaN
4,S001,HW1,1,HW1_PCA_Q05,MCQ,PCA Q05,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,B,B,NaN


- Replace score 0/1 scores with correct, incorrect => First Attempt
- Create column that counts the number of incorrect attemps => Incorrects
- Create column that counts the number of correct attemps => Corrects
- Create column that counts the number of hints => Hints
- Rename columns to have correct names

In [293]:
scores = [
    (student_observations['score']==student_observations['max_score'], 'correct'),
    (student_observations['score']!=student_observations['max_score'], 'incorrect')
]
student_observations['First Attempt']=student_observations['score'].case_when(scores)
student_observations["Incorrects"] = (student_observations["First Attempt"] == "incorrect").astype(int)
student_observations["Corrects"]   = (student_observations["First Attempt"] == "correct").astype(int)
student_observations["Hints"]      = 0
student_info = student_observations[['student_id','source_question','observation_id','First Attempt',"Incorrects", "Corrects","Hints",'primary_kc_id']].rename(columns={'student_id':'Anon Student Id',
                                                                                                                                              'source_question':'Problem Name',
                                                                                                                                              'observation_id':'Step Name'})

### Class plan

In [294]:
class_plan=class_plan[['class_date']]
class_plan.head()

,class_date
class_num,
1,2025-09-05
2,2025-09-12
3,2025-09-19
4,2025-09-26
5,2025-10-03


- Create a time column that combines date and time by using sequence

In [295]:
hwk_items=hwk_items[['sequence','observation_id']]

In [296]:
date = hwk_items.join(class_plan)

In [297]:
date['Time'] = (date['class_date'] + pd.to_timedelta(date["sequence"], unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")

In [298]:
date

,sequence,observation_id,class_date,Time
class_num,,,,
1,1,HW1_PCA_Q01,2025-09-05,2025-09-05 00:00:01
1,2,HW1_PCA_Q02,2025-09-05,2025-09-05 00:00:02
1,3,HW1_PCA_Q03,2025-09-05,2025-09-05 00:00:03
1,4,HW1_PCA_Q04,2025-09-05,2025-09-05 00:00:04
1,5,HW1_PCA_Q05,2025-09-05,2025-09-05 00:00:05
...,...,...,...,...
27,56,MF2_FRQ6A,2026-03-13,2026-03-13 00:00:56
27,57,MF2_FRQ6B,2026-03-13,2026-03-13 00:00:57
27,58,MF2_FRQ6C,2026-03-13,2026-03-13 00:00:58


In [299]:
date_merge = pd.merge(
    student_info,
    date[['observation_id','Time']],
    left_on="Step Name",
    right_on="observation_id",
    how="left"
)
date_merge.drop(columns='observation_id')

,Anon Student Id,Problem Name,Step Name,First Attempt,Incorrects,Corrects,Hints,primary_kc_id,Time
0,S001,PCA Q01,HW1_PCA_Q01,incorrect,1,0,0,KC.U1.02.observational_unit_variable,2025-09-05 00:00:01
1,S001,PCA Q02,HW1_PCA_Q02,correct,0,1,0,KC.U1.03.variable_type_cat_quant,2025-09-05 00:00:02
2,S001,PCA Q03,HW1_PCA_Q03,incorrect,1,0,0,KC.U1.03.variable_type_cat_quant,2025-09-05 00:00:03
3,S001,PCA Q04,HW1_PCA_Q04,correct,0,1,0,KC.U1.05.categorical_freq_relative,2025-09-05 00:00:04
4,S001,PCA Q05,HW1_PCA_Q05,correct,0,1,0,KC.U1.05.categorical_freq_relative,2025-09-05 00:00:05
...,...,...,...,...,...,...,...,...,...
21803,S025,FRQ 6,MF2_FRQ6A,incorrect,1,0,0,KC.U1.09.quantitative_display_reading,2026-03-13 00:00:56
21804,S025,FRQ 6,MF2_FRQ6B,correct,0,1,0,KC.U7.12.mean_hypotheses,2026-03-13 00:00:57
21805,S025,FRQ 6,MF2_FRQ6C,incorrect,1,0,0,KC.U7.13.t_test_conditions,2026-03-13 00:00:58
21806,S025,FRQ 6,MF2_FRQ6D,correct,0,1,0,KC.U7.14.t_test_statistic_mean,2026-03-13 00:00:59


### KC Nodes

In [300]:
unit_merge = pd.merge(
    date_merge,
    kc_nodes[['kc_id','kc_label','reporting_group']],
    left_on="primary_kc_id",
    right_on="kc_id",
    how="left"
)
unit_merge

,Anon Student Id,Problem Name,Step Name,First Attempt,Incorrects,Corrects,Hints,primary_kc_id,observation_id,Time,kc_id,kc_label,reporting_group
0,S001,PCA Q01,HW1_PCA_Q01,incorrect,1,0,0,KC.U1.02.observational_unit_variable,HW1_PCA_Q01,2025-09-05 00:00:01,KC.U1.02.observational_unit_variable,Observational unit and variable,Unit 1: Data foundations
1,S001,PCA Q02,HW1_PCA_Q02,correct,0,1,0,KC.U1.03.variable_type_cat_quant,HW1_PCA_Q02,2025-09-05 00:00:02,KC.U1.03.variable_type_cat_quant,Categorical vs quantitative variable,Unit 1: Data foundations
2,S001,PCA Q03,HW1_PCA_Q03,incorrect,1,0,0,KC.U1.03.variable_type_cat_quant,HW1_PCA_Q03,2025-09-05 00:00:03,KC.U1.03.variable_type_cat_quant,Categorical vs quantitative variable,Unit 1: Data foundations
3,S001,PCA Q04,HW1_PCA_Q04,correct,0,1,0,KC.U1.05.categorical_freq_relative,HW1_PCA_Q04,2025-09-05 00:00:04,KC.U1.05.categorical_freq_relative,Frequency and relative-frequency tables,Unit 1: Categorical data
4,S001,PCA Q05,HW1_PCA_Q05,correct,0,1,0,KC.U1.05.categorical_freq_relative,HW1_PCA_Q05,2025-09-05 00:00:05,KC.U1.05.categorical_freq_relative,Frequency and relative-frequency tables,Unit 1: Categorical data
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21803,S025,FRQ 6,MF2_FRQ6A,incorrect,1,0,0,KC.U1.09.quantitative_display_reading,MF2_FRQ6A,2026-03-13 00:00:56,KC.U1.09.quantitative_display_reading,Read quantitative displays,Unit 1: Quantitative displays
21804,S025,FRQ 6,MF2_FRQ6B,correct,0,1,0,KC.U7.12.mean_hypotheses,MF2_FRQ6B,2026-03-13 00:00:57,KC.U7.12.mean_hypotheses,Hypotheses for population mean,Unit 7: Tests for means
21805,S025,FRQ 6,MF2_FRQ6C,incorrect,1,0,0,KC.U7.13.t_test_conditions,MF2_FRQ6C,2026-03-13 00:00:58,KC.U7.13.t_test_conditions,t-test conditions for means,Unit 7: Tests for means
21806,S025,FRQ 6,MF2_FRQ6D,correct,0,1,0,KC.U7.14.t_test_statistic_mean,MF2_FRQ6D,2026-03-13 00:00:59,KC.U7.14.t_test_statistic_mean,t-test statistic for a mean,Unit 7: Tests for means


- Add columns with KC labels, and problem hierarchy
- Create an opportunity column that counts how many times a student has seen a particular KC so far in the sequence.

In [301]:
datashop = unit_merge.drop(columns=['observation_id','kc_id','primary_kc_id']).rename(columns={'kc_label' : 'KC (Default)', 'reporting_group' : 'Problem Hierarchy'})
datashop = datashop.sort_values(["Anon Student Id", "Time"])
datashop["Opportunity (Default)"] = (
    datashop.groupby(["Anon Student Id", "KC (Default)"]).cumcount() + 1
)

datashop

,Anon Student Id,Problem Name,Step Name,First Attempt,Incorrects,Corrects,Hints,Time,KC (Default),Problem Hierarchy,Opportunity (Default)
0,S001,PCA Q01,HW1_PCA_Q01,incorrect,1,0,0,2025-09-05 00:00:01,Observational unit and variable,Unit 1: Data foundations,1
1,S001,PCA Q02,HW1_PCA_Q02,correct,0,1,0,2025-09-05 00:00:02,Categorical vs quantitative variable,Unit 1: Data foundations,1
2,S001,PCA Q03,HW1_PCA_Q03,incorrect,1,0,0,2025-09-05 00:00:03,Categorical vs quantitative variable,Unit 1: Data foundations,2
3,S001,PCA Q04,HW1_PCA_Q04,correct,0,1,0,2025-09-05 00:00:04,Frequency and relative-frequency tables,Unit 1: Categorical data,1
4,S001,PCA Q05,HW1_PCA_Q05,correct,0,1,0,2025-09-05 00:00:05,Frequency and relative-frequency tables,Unit 1: Categorical data,2
...,...,...,...,...,...,...,...,...,...,...,...
21805,S025,FRQ 6,MF2_FRQ6C,incorrect,1,0,0,2026-03-13 00:00:58,t-test conditions for means,Unit 7: Tests for means,4
21746,S025,Investigative Q1,MF1_FRQ6D,incorrect,1,0,0,2026-03-13 00:00:59,Bootstrap percentile cutoffs,Unit 10: Bootstrapping and Power,3
21806,S025,FRQ 6,MF2_FRQ6D,correct,0,1,0,2026-03-13 00:00:59,t-test statistic for a mean,Unit 7: Tests for means,7
21747,S025,Investigative Q1,MF1_FRQ6E,incorrect,1,0,0,2026-03-13 00:01:00,Interpret a bootstrap interval in context,Unit 10: Bootstrapping and Power,3


In [302]:
out = datashop[[
    "Anon Student Id",
    "Problem Hierarchy",
    "Problem Name",
    "Step Name",
    "First Attempt",
    "Incorrects",
    "Corrects",
    "Hints",
    "KC (Default)",
    "Opportunity (Default)",
]]

In [303]:
out.to_csv('../data/processed/test_student_step.txt', sep='\t', index=False)

In [304]:
out

,Anon Student Id,Problem Hierarchy,Problem Name,Step Name,First Attempt,Incorrects,Corrects,Hints,KC (Default),Opportunity (Default)
0,S001,Unit 1: Data foundations,PCA Q01,HW1_PCA_Q01,incorrect,1,0,0,Observational unit and variable,1
1,S001,Unit 1: Data foundations,PCA Q02,HW1_PCA_Q02,correct,0,1,0,Categorical vs quantitative variable,1
2,S001,Unit 1: Data foundations,PCA Q03,HW1_PCA_Q03,incorrect,1,0,0,Categorical vs quantitative variable,2
3,S001,Unit 1: Categorical data,PCA Q04,HW1_PCA_Q04,correct,0,1,0,Frequency and relative-frequency tables,1
4,S001,Unit 1: Categorical data,PCA Q05,HW1_PCA_Q05,correct,0,1,0,Frequency and relative-frequency tables,2
...,...,...,...,...,...,...,...,...,...,...
21805,S025,Unit 7: Tests for means,FRQ 6,MF2_FRQ6C,incorrect,1,0,0,t-test conditions for means,4
21746,S025,Unit 10: Bootstrapping and Power,Investigative Q1,MF1_FRQ6D,incorrect,1,0,0,Bootstrap percentile cutoffs,3
21806,S025,Unit 7: Tests for means,FRQ 6,MF2_FRQ6D,correct,0,1,0,t-test statistic for a mean,7
21747,S025,Unit 10: Bootstrapping and Power,Investigative Q1,MF1_FRQ6E,incorrect,1,0,0,Interpret a bootstrap interval in context,3


# Learnsphere Outputs

Results of the Learnsphere BKT model :

In [ ]:
import xmltodict

with open("../data/outputs/Model-values.xml", "r") as f:
    data = xmltodict.parse(f.read())

model = data['model_values']['model']

print("=" * 40)
print(f"  BKT Model Results — {model['name']}")
print("=" * 40)
print(f"  Log Likelihood        : {model['log_likelihood']}")
print(f"  AIC                   : {model['AIC']}")
print(f"  BIC                   : {model['BIC']}")
print("-" * 40)
print(f"  RMSE                  : {model['RMSE']}")
print(f"  Accuracy              : {model['accuracy']}")
print("-" * 40)
print(f"  CV Student Stratified : {model['cv_student_stratified']}")
print(f"  CV Item Stratified    : {model['cv_item_stratified']}")
print(f"  CV Non-Stratified     : {model['cv_non_stratified']}")
print("=" * 40)

  BKT Model Results — Default
  Log Likelihood        : 14776.2502
  AIC                   : 31432.5003
  BIC                   : 38943.1305
----------------------------------------
  RMSE                  : 0.4920
  Accuracy              : 0.5834
----------------------------------------
  CV Student Stratified : 0.5006
  CV Item Stratified    : 0.5019
  CV Non-Stratified     : 0.4948


In [314]:
with open("../data/outputs/Parameter-estimate-values.xml", "r") as f:
    data = xmltodict.parse(f.read())

params = data['parameters']['parameter']

print("=" * 75)
print(f"  BKT Parameter Estimates — {len(params)} skills")
print("=" * 75)
print(f"  {'ID':<4} {'Skill Name':<40} {'P-Init':>7} {'P-Learn':>8} {'P-Slip':>7} {'P-Guess':>8}")
print("-" * 75)

for p in params:
    print(f"  {p['id']:<4} {p['name']:<40} {float(p['intercept']):>7.4f} {float(p['slope']):>8.4f} {float(p['slip']):>7.4f} {float(p['guess']):>8.4f}")

print("=" * 75)

  BKT Parameter Estimates — 235 skills
  ID   Skill Name                                P-Init  P-Learn  P-Slip  P-Guess
---------------------------------------------------------------------------
  1    Observational unit and variable           0.4451   0.0508  0.0134   0.3000
  2    Categorical vs quantitative variable      0.9204   0.0180  0.2615   0.3000
  3    Frequency and relative-frequency tables   0.6345   0.0110  0.2252   0.3000
  4    Bar, pie, and segmented-bar displays      0.8311   0.1678  0.3000   0.2755
  5    Two-way categorical association           0.3835   0.0150  0.1082   0.3000
  6    Choose categorical display                0.4896   0.0718  0.0103   0.3000
  7    Sample, population, and scope of generalization  0.2037   0.0079  0.0150   0.3000
  8    Discrete vs continuous quantitative variable  0.6934   0.0031  0.2171   0.3000
  9    Read quantitative displays                0.2075   0.0087  0.0115   0.3000
  10   Shape, clusters, gaps, and outliers       0.590